In [1]:
!nvidia-smi

Thu Oct  8 07:06:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from numba import cuda
import os

device = cuda.get_current_device()

print(f"Device Name: {device.name}")
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max Block Dimensions: {device.MAX_BLOCK_DIM_X}, {device.MAX_BLOCK_DIM_Y}, {device.MAX_BLOCK_DIM_Z}")
print(f"Warp Size: {device.WARP_SIZE}")

Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count (SMs): 40
Max Threads Per Block: 1024
Max Block Dimensions: 1024, 1024, 64
Warp Size: 32


In [6]:
import numpy as np
import time
from numba import cuda

# CPU Reference Implementation
def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

# GPU Kernel Implementation
@cuda.jit
def double_gpu_kernel(d_in, d_out):
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

In [7]:
def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)

    # CPU Measurement
    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan')

    # GPU Setup
    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    # Warmup
    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()

    # GPU Total Time
    t0 = time.perf_counter()

    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)

    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)

    h_out = d_out.copy_to_host()
    cuda.synchronize()

    gpu_total_ms = (time.perf_counter() - t0) * 1000.0

    # GPU Kernel Only Time
    t0 = time.perf_counter()

    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()

    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0

    return cpu_time_ms, gpu_total_ms, gpu_compute_ms

In [8]:
sizes = [
    100,
    1000,
    10000,
    100000,
    1000000,
    5000000,
    10000000
]

print(f"{'N':<12} {'CPU (ms)':<15} {'GPU Total (ms)':<18} {'GPU Kernel (ms)':<18} {'Winner'}")
print("-" * 80)

results = []

for N in sizes:
    cpu, gpu_total, gpu_kernel = benchmark_run(N)

    if np.isnan(cpu):
        winner = "N/A"
    else:
        winner = "GPU" if gpu_total < cpu else "CPU"

    results.append((N, cpu, gpu_total, gpu_kernel, winner))

    print(f"{N:<12} {cpu:<15.4f} {gpu_total:<18.4f} {gpu_kernel:<18.4f} {winner}")

N            CPU (ms)        GPU Total (ms)     GPU Kernel (ms)    Winner
--------------------------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


100          0.0653          1.3817             0.1029             CPU
1000         0.2571          0.5515             0.0741             CPU
10000        2.3653          0.6112             0.0578             GPU
100000       24.0440         0.7081             0.0679             GPU
1000000      249.8879        4.6170             0.1118             GPU
5000000      nan             13.2103            0.2590             N/A
10000000     nan             24.6483            0.4712             N/A


In [9]:
# DEFECTIVE KERNEL
@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)

    # Missing boundary check!
    d_arr[idx] = 999.0


# TEST HARNESS
N = 1000

h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

threads_per_block = 256
blocks_per_grid = 4

print("Launching defective kernel with 1024 threads on 1000-element array...")

defective_kernel[blocks_per_grid, threads_per_block](d_arr)

cuda.synchronize()

print("Execution finished (or did it crash?). Checking host copy...")

h_result = d_arr.copy_to_host()

print(f"Elements processed: {len(h_result)}")
print(f"Last element: {h_result[-1]}")

Launching defective kernel with 1024 threads on 1000-element array...
Execution finished (or did it crash?). Checking host copy...
Elements processed: 1000
Last element: 999.0


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [10]:
@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):

    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)

    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0


# Configuration
N = 37
threads_per_block = 8

blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

# Allocate GPU memory
log_b = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_t = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_g = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_a = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)

# Launch GPU kernel
inspect_threads_kernel[blocks_per_grid, threads_per_block](
    log_b, log_t, log_g, log_a, N
)

cuda.synchronize()

# Copy results to CPU
host_b = log_b.copy_to_host()
host_t = log_t.copy_to_host()
host_g = log_g.copy_to_host()
host_a = log_a.copy_to_host()

# Display results
print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<15}")
print("-" * 60)

for i in range(32, 40):
    status = "ACTIVE" if host_a[i] == 1 else "IDLE (GUARDED)"

    print(f"{host_g[i]:<10} | {host_b[i]:<10} | {host_t[i]:<10} | {status:<15}")

Global ID  | Block ID   | Thread ID  | Status         
------------------------------------------------------------
32         | 4          | 0          | ACTIVE         
33         | 4          | 1          | ACTIVE         
34         | 4          | 2          | ACTIVE         
35         | 4          | 3          | ACTIVE         
36         | 4          | 4          | ACTIVE         
37         | 4          | 5          | IDLE (GUARDED) 
38         | 4          | 6          | IDLE (GUARDED) 
39         | 4          | 7          | IDLE (GUARDED) 


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [11]:
from numba import cuda
import numpy as np
import time

# BUGGY NAIVE SUMMATION
@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)

    if idx < d_arr.shape[0]:
        # DATA RACE
        d_total[0] += d_arr[idx]


# CORRECT ATOMIC SUMMATION
@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)

    if idx < d_arr.shape[0]:
        # Atomic operation prevents race conditions
        cuda.atomic.add(d_total, 0, d_arr[idx])

In [12]:
N = 50000

h_arr = np.ones(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

print(f"{'Trial':<10} {'Target Sum':<15} {'Naive Output':<20} {'Lost Updates'}")
print("-" * 65)

naive_results = []

for trial in range(1, 6):

    d_total = cuda.to_device(np.zeros(1, dtype=np.float32))

    naive_sum_kernel[blocks_per_grid, threads_per_block](d_arr, d_total)

    cuda.synchronize()

    result = d_total.copy_to_host()[0]

    lost_updates = 50000.0 - result

    naive_results.append(result)

    print(f"{trial:<10} {50000.0:<15.1f} {result:<20.1f} {lost_updates:.1f}")

Trial      Target Sum      Naive Output         Lost Updates
-----------------------------------------------------------------
1          50000.0         2.0                  49998.0
2          50000.0         2.0                  49998.0
3          50000.0         2.0                  49998.0
4          50000.0         2.0                  49998.0
5          50000.0         2.0                  49998.0


In [13]:
# Prepare data
N = 50000

h_arr = np.ones(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

# Warmup both kernels
d_total = cuda.to_device(np.zeros(1, dtype=np.float32))

naive_sum_kernel[blocks_per_grid, threads_per_block](d_arr, d_total)
cuda.synchronize()

d_total = cuda.to_device(np.zeros(1, dtype=np.float32))

atomic_sum_kernel[blocks_per_grid, threads_per_block](d_arr, d_total)
cuda.synchronize()


# Measure Naive Kernel
d_total_naive = cuda.to_device(np.zeros(1, dtype=np.float32))

t0 = time.perf_counter()

naive_sum_kernel[blocks_per_grid, threads_per_block](d_arr, d_total_naive)
cuda.synchronize()

naive_time_ms = (time.perf_counter() - t0) * 1000.0


# Measure Atomic Kernel
d_total_atomic = cuda.to_device(np.zeros(1, dtype=np.float32))

t0 = time.perf_counter()

atomic_sum_kernel[blocks_per_grid, threads_per_block](d_arr, d_total_atomic)
cuda.synchronize()

atomic_time_ms = (time.perf_counter() - t0) * 1000.0


# Results
atomic_result = d_total_atomic.copy_to_host()[0]

print("=" * 50)
print(f"Expected Sum: 50000.0")
print(f"Atomic Sum Output: {atomic_result}")
print(f"Correct Result: {'YES' if atomic_result == 50000.0 else 'NO'}")

print("-" * 50)

print(f"Naive Kernel Time: {naive_time_ms:.4f} ms")
print(f"Atomic Kernel Time: {atomic_time_ms:.4f} ms")

print("=" * 50)

Expected Sum: 50000.0
Atomic Sum Output: 50000.0
Correct Result: YES
--------------------------------------------------
Naive Kernel Time: 0.4052 ms
Atomic Kernel Time: 0.4585 ms


In [14]:
import math
import numpy as np
from numba import cuda

@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):

    # Retrieve 2D spatial coordinates
    x, y = cuda.grid(2)

    if x < width and y < height:

        dx = float(x - center_x)
        dy = float(y - center_y)

        dist = math.sqrt(dx * dx + dy * dy)

        max_dist = math.sqrt(float(center_x**2 + center_y**2))

        intensity = 1.0 - (dist / max_dist)

        if intensity < 0.0:
            intensity = 0.0

        d_canvas[y, x] = intensity


# 2D EXECUTION CONFIGURATION
WIDTH, HEIGHT = 1024, 1024

canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)

d_canvas = cuda.to_device(canvas)

# 2D Blocks (16x16 = 256 threads)
threads_2d = (16, 16)

blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1]
)

# Launch kernel
radial_vignette_2d[blocks_2d, threads_2d](
    d_canvas,
    WIDTH,
    HEIGHT,
    WIDTH // 2,
    HEIGHT // 2
)

cuda.synchronize()

# Copy results to CPU
result_canvas = d_canvas.copy_to_host()

# Display results
print("=" * 50)

print(f"Generated 2D matrix shape: {result_canvas.shape}")

print(f"Grid Dimensions: {blocks_2d}")

print(f"Total Blocks: {blocks_2d[0] * blocks_2d[1]}")

print(f"Center Pixel (512, 512): {result_canvas[512, 512]:.3f}")

print(f"Corner Pixel (0, 0): {result_canvas[0, 0]:.3f}")

print("=" * 50)

Generated 2D matrix shape: (1024, 1024)
Grid Dimensions: (64, 64)
Total Blocks: 4096
Center Pixel (512, 512): 1.000
Corner Pixel (0, 0): 0.000


In [15]:
import hashlib

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):

    h = hashlib.sha256()

    # Student ID
    h.update(student_id_str.strip().encode('utf-8'))

    # GPU Name
    device_name = cuda.get_current_device().name

    if isinstance(device_name, str):
        device_name = device_name.encode('utf-8')

    h.update(device_name)

    # GPU Computation Result
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())

    # Crossover Point
    h.update(str(exp_table_crossover).encode('utf-8'))

    digest = h.hexdigest()[:16].upper()

    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)

    return digest


# Your actual Student ID
STUDENT_ID = "230103082"

# Your empirical crossover point
CROSSOVER_N = 10000

# Generate token
verification_token = generate_lab_checksum(
    STUDENT_ID,
    d_canvas,
    CROSSOVER_N
)

OFFICIAL VERIFICATION CHECKSUM TOKEN: 984DB1C1B559734D
